# AG News Comparative Study
## CS371N Final Project — Spring 2026

---

## Dataset

**AG News** is a collection of over 1 million news articles gathered from more than 2,000 news sources by the academic news search engine ComeToMyHead over the course of more than a year.

Each example is a `(news article, label)` pair classified into one of four categories:

| ID | Label    |
|----|----------|
| 0  | World    |
| 1  | Sports   |
| 2  | Business |
| 3  | Sci/Tech |

---

## Models

We implement and compare five modeling approaches on the same dataset:

| # | Approach | Description |
|---|----------|-------------|
| 1 | **TF-IDF + Logistic Regression** | Classical baseline using n-gram features with a linear classifier |
| 2 | **Static Embeddings + MLP Classifier** | Pretrained Word2Vec embeddings with a simple MLP Classier on top |
| 3 | **LSTM (trained from scratch)** | Neural model capturing sequential structure without pretraining |
| 4 | **DistilBERT (fine-tuned)** | Pretrained transformer with task-specific supervised fine-tuning |
| 5 | **Zero-shot** | Pretrained model evaluated without any task-specific training |

---

## Evaluation

All models are evaluated on the **same held-out test split** using:
- **Accuracy**
- **Macro F1**

---

## Reproducibility

To reproduce results:

1. Clone the repo
2. Run `ag_news_comparative_study.ipynb` top to bottom

**Authors:** Anh Nguyen, Berke Kara

Acknowledgement:
We appreciate fancyzhx for creating and sharing the AG News dataset for research.

#Pre-Process

In [ ]:
!pip install gensim datasets

from collections import Counter
from datasets import load_dataset,concatenate_datasets,Dataset, DatasetDict
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, KeyedVectors
from gensim.utils import simple_preprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import pipeline
from transformers_interpret import SequenceClassificationExplainer


In [ ]:
ds = load_dataset("fancyzhx/ag_news")
ds_combined = concatenate_datasets([ds["train"], ds["test"]])
df = pd.DataFrame(ds_combined)
w2v_model = api.load("word2vec-google-news-300")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline_device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {device}")

# Global Label Mappings
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {v: k for k, v in id2label.items()}

In [ ]:
df_sampled, _ = train_test_split(
    df,
    train_size=30000,
    random_state=42,
    stratify=df["label"]
)

train_df, temp_df = train_test_split(
    df_sampled,
    test_size=0.3,
    random_state=42,
    stratify=df_sampled["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

In [ ]:
X_train = train_df['text']
y_train = train_df['label']

X_val = val_df['text']
y_val = val_df['label']

X_test = test_df['text']
y_test = test_df['label']

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_val)}")
print(f"Test examples: {len(X_test)}")

# Model 1: Classical Baseline

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

val_preds = model.predict(X_val_vec)
test_preds = model.predict(X_test_vec)

print("Model 1: Classical Baseline Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(y_val, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=list(id2label.values()), digits=4))

# Model 2: Pretrained Word2Vec with Classifier

In [ ]:
def text_to_vector(text, model, dim=300):
    """
    Convert a sentence/document to a fixed-size vector
    by averaging all word vectors (mean pooling).
    """
    tokens = simple_preprocess(text)

    valid_vectors = [model[token] for token in tokens if token in model]

    if not valid_vectors:
        return np.zeros(dim)

    return np.mean(valid_vectors, axis=0)


In [ ]:
X = np.array([text_to_vector(text, w2v_model) for text in X_train])
y = np.array(y_train)

clf = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=1000)

clf.fit(X, y)
print("Classifier trained")


In [ ]:
X_test_vec = np.array([text_to_vector(text, w2v_model) for text in X_test])
X_val_vec = np.array([text_to_vector(text, w2v_model) for text in X_val])

val_preds = clf.predict(X_val_vec)
test_preds = clf.predict(X_test_vec)

print("Model 2: Pretrained Word2Vec + Classifier Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(y_val, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=list(id2label.values()), digits=4))

# Model 3: LSTM from Scratch

In [ ]:
from torch.nn.utils.rnn import pad_sequence

# 1. Build Vocabulary
def build_vocab(texts, max_size=20000):
    counter = Counter()
    for text in texts:
        counter.update(simple_preprocess(text))

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(X_train)
print(f"Vocabulary size: {len(vocab)}")

# 2. Dataset Class
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = simple_preprocess(self.texts[idx])
        tokens = tokens[:self.max_length]
        indices = [self.vocab.get(token, self.vocab['<UNK>']) for token in tokens]
        if len(indices) == 0:
            indices = [self.vocab['<UNK>']] # Handle empty sequences
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# 3. Collate Function (for padding)
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    # Pad sequences to match the longest one in the batch
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_sequences, lengths, torch.stack(labels)

# 4. Create DataLoaders
train_dataset = TextDataset(X_train, y_train, vocab)
val_dataset = TextDataset(X_val, y_val, vocab)
test_dataset = TextDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Created train_loader with {len(train_loader)} batches.")
print(f"Created val_loader with {len(val_loader)} batches.")
print(f"Created test_loader with {len(test_loader)} batches.")

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))
        # embedded: (batch, seq_len, embed_dim)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (hidden, _) = self.lstm(packed)

        # hidden: (num_layers * 2, batch, hidden_dim)
        forward_hidden  = hidden[-2]   # last layer, forward
        backward_hidden = hidden[-1]   # last layer, backward

        # Concatenate both directions
        combined = torch.cat([forward_hidden, backward_hidden], dim=1)
        # combined: (batch, hidden_dim * 2)

        out = self.dropout(combined)
        # (batch, num_classes)
        return self.fc(out)

model = LSTMClassifier(
    vocab_size  = len(vocab),
    embed_dim   = 128,
    hidden_dim  = 256,
    num_classes = 4,
    num_layers  = 2,
    dropout     = 0.3
).to(device)

# Count parameters — report this in your paper
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for sequences, lengths, labels in loader:
        sequences, lengths, labels = (
            sequences.to(device),
            lengths.to(device),
            labels.to(device)
        )

        optimizer.zero_grad()
        outputs = model(sequences, lengths)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping — important for LSTM stability
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for sequences, lengths, labels in loader:
            sequences, lengths, labels = (
                sequences.to(device),
                lengths.to(device),
                labels.to(device)
            )
            outputs = model(sequences, lengths)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

# Train
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion) # Changed to val_loader
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

In [ ]:
def get_preds(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for sequences, lengths, labels in loader:
            outputs = model(sequences.to(device), lengths.to(device))
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return all_labels, all_preds

val_labels, val_preds = get_preds(model, val_loader)
test_labels, test_preds = get_preds(model, test_loader)

print("Model 3: LSTM from Scratch Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(val_labels, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(val_labels, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds,
      target_names=list(id2label.values()), digits=4))

# Model 4: Pretrained Model with Fine-Tuning
---



In [ ]:
import datasets # Import module to avoid conflict with torch.utils.data.Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

dataset = datasets.DatasetDict({
    "train": datasets.Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "validation": datasets.Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False),
    "test": datasets.Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./distilbert_agnews",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()

# val results
predictions = trainer.predict(tokenized_dataset["validation"])
val_logits = predictions.predictions
val_labels = predictions.label_ids
val_preds = np.argmax(val_logits, axis=1)

# test results
test_preds_output = trainer.predict(tokenized_dataset["test"])
test_logits = test_preds_output.predictions
test_labels = test_preds_output.label_ids
test_preds = np.argmax(test_logits, axis=1)

print("Model 4: Pretrained Model with Fine-Tuning Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(val_labels, val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(val_labels, val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(test_labels, test_preds):.4f}")
print(f"Test Macro F1: {f1_score(test_labels, test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=list(id2label.values()), digits=4))

In [ ]:
!pip install transformers-interpret captum

### Interpretability with Integrated Gradients

Integrated Gradients for our DistilBERT model. Attribution score for each word, positive scores push toward the predicted class, negative scores push away.

In [ ]:
model.config.id2label = id2label
model.config.label2id = label2id

cls_explainer = SequenceClassificationExplainer(
    model,
    tokenizer
)

# Take the first example from the test set
sample_text = test_df["text"].iloc[0]
true_label = test_df["label"].iloc[0]

print(f"Original Text: {sample_text}")
print(f"True Category: {id2label[true_label]}\n")

# Calculate attributions
word_attributions = cls_explainer(sample_text)

# Visualize the attributions in an HTML format directly in Colab
cls_explainer.visualize()

# Model 5: pretrained model without task-specific training

In [ ]:
candidate_labels = list(label2id.keys())

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=pipeline_device
)

def predict_zero_shot(texts, batch_size=16):
    preds = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        outputs = zero_shot_classifier(
            batch,
            candidate_labels=candidate_labels,
            hypothesis_template="This news article is about {}.",
            multi_label=False
        )

        if isinstance(outputs, dict):
            outputs = [outputs]

        for output in outputs:
            pred_label = output["labels"][0]
            preds.append(label2id[pred_label])

        print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)} examples")

    return np.array(preds)

In [ ]:
val_texts = X_val.tolist()
val_labels = y_val.to_numpy()

zero_val_preds = predict_zero_shot(val_texts, batch_size=32)

In [ ]:
test_texts = X_test.tolist()
test_labels = y_test.to_numpy()

zero_test_preds = predict_zero_shot(test_texts, batch_size=16)

print("Model 5: Zero-Shot Classification Results")
print("-" * 50)
print(f"Validation Accuracy: {accuracy_score(val_labels, zero_val_preds):.4f}")
print(f"Validation Macro F1: {f1_score(val_labels, zero_val_preds, average='macro'):.4f}")
print(f"Test Accuracy: {accuracy_score(test_labels, zero_test_preds):.4f}")
print(f"Test Macro F1: {f1_score(test_labels, zero_test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(
    test_labels,
    zero_test_preds,
    target_names=list(id2label.values()),
    digits=4
))

### Error Analysis (Best Model: DistilBERT)


In [ ]:
# test_preds currently holds the predictions from Model 4 (DistilBERT)
# test_labels holds the true labels
incorrect_indices = np.where(test_preds != test_labels)[0]

# Create a DataFrame to easily view the errors using global id2label
error_data = {
    "Text": [test_df["text"].iloc[i] for i in incorrect_indices],
    "True_Label": [id2label[test_labels[i]] for i in incorrect_indices],
    "Predicted_Label": [id2label[test_preds[i]] for i in incorrect_indices]
}

error_df = pd.DataFrame(error_data)

print(f"Total misclassifications by DistilBERT: {len(error_df)} out of {len(test_labels)}")
print("Displaying the first 20 errors:\n")
print("-"*50)
# Set pandas display options so text isn't truncated
pd.set_option('display.max_colwidth', None)
display(error_df.head(20))

# Reset display options afterward
pd.reset_option('display.max_colwidth')

### Error Analysis


**1. Blurring between Business and Sci/Tech:**
- **Explanation:** The model struggles significantly when technology companies are involved in business activities (like lawsuits, product launches, or acquisitions). It tends to group tech-related keywords (like "Java", "DVD", "PCs", "Amazon", "Kodak") and misclassifies purely business oriented actions as Sci/Tech, or vice-versa when money/lawsuits are mentioned in a tech context.
- **Examples: **
  - *"Kodak Seeks \$1 Billion from Sun in Java Suit"* (True: Sci/Tech, Pred: Business/World)
  - *"Amazon launching DVD rentals"* (True: Business, Pred: Sci/Tech)
  - *"Lessons Airlines Can Learn From PCs"* (True: Business, Pred: Sci/Tech)

**2. Blurring between Business and World News:**
- **Explanation:** Macroeconomic indicators, government policies affecting markets, and national employment statistics blur the line between Business and World News. The model sometimes misinterprets global economic updates as general World news because they deal with countries and national statistics rather than specific corporate entities.
- **Examples:**
  - *"US economy generated 144,000 jobs in August (AFP)"* (True: Business, Pred: World)

**3. Keyword reliance / Lack of deep context:**
- **Explanation:** DistilBERT sometimes acts like a bag-of-words model, swayed by strong domain specific vocabulary even when the actual syntactic meaning of the sentence points to a different category. Metaphorical language or cross-domain topics confuse the classifier.
- **Examples:**
  - *"Ethics code written to reprogram tech industry"* (True: Business, Pred: Sci/Tech - confused by "reprogram" and "tech industry")